# NEW EXAMPLE: County-Level Child Poverty Analysis with AI (Python)

**Estimated time:** 20 minutes  
**Packages needed:** `pandas`, `matplotlib`, `seaborn`  
**Quick run:** Run all cells in Jupyter or execute `jupyter nbconvert --execute ai-exercise-new-example.ipynb`  

## Human+AI Workflow Overview

This exercise demonstrates a complete human+AI data analysis workflow for county-level child poverty data using Python. You'll work step-by-step with an AI assistant to load, analyze, and visualize the data.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Read the dataset
child_poverty = pd.read_csv('toy-data/child-poverty-county.csv')

print("Dataset loaded successfully!")
print(f"Shape: {child_poverty.shape}")

## Human+AI Workflow Steps

### Step 1: Data Exploration and Setup

**Your role:** Ask AI to help you explore and understand the dataset.

**Barebones prompt (copy-paste):**
```
Explore toy-data/child-poverty-county.csv and summarize the data structure, key variables, and any data quality issues.
```

**Expanded prompt:**
```
Load and explore the child poverty county dataset using pandas. Please provide a summary of the data structure, describe what each column represents, identify any missing values or outliers, and suggest what analysis might be appropriate for this data.
```

**Expected AI guidance:** AI should identify the GEOID, NAME, state, children_in_poverty, total_children_under18, and child_poverty_rate_pct columns and suggest county-level analysis.

In [ ]:
# Explore the data structure
print("Data Info:")
print(child_poverty.info())
print("\nFirst few rows:")
print(child_poverty.head())
print("\nSummary statistics:")
print(child_poverty.describe())

# Check for missing values
print("\nMissing values:")
print(child_poverty.isnull().sum())

# Preview top counties by poverty rate
print("\nTop 10 counties by poverty rate:")
top_counties = child_poverty.nlargest(10, 'child_poverty_rate_pct')
print(top_counties[['NAME', 'state', 'child_poverty_rate_pct']])

### Step 2: Ask AI to Compute and Validate Indicators

**Your role:** Ask AI to compute additional indicators and validate the data.

**Barebones prompt (copy-paste):**
```
Compute county child poverty rate, show top 10 counties, and validate calculations match the provided rates.
```

**Expanded prompt:**
```
Using the child poverty dataset, compute the child poverty rate from the raw counts and compare it to the provided rate column. Show the top 10 counties with highest poverty rates and verify our calculations are correct.
```

In [ ]:
# Compute poverty rate and validate
child_poverty_validated = child_poverty.copy()
child_poverty_validated['computed_poverty_rate'] = np.round(
    100 * child_poverty_validated['children_in_poverty'] / child_poverty_validated['total_children_under18'], 1
)
child_poverty_validated['rate_difference'] = np.abs(
    child_poverty_validated['child_poverty_rate_pct'] - child_poverty_validated['computed_poverty_rate']
)

# Check validation
validation_issues = child_poverty_validated[child_poverty_validated['rate_difference'] > 0.1]
print("Validation issues (rate differences > 0.1%):")
print(validation_issues[['NAME', 'state', 'child_poverty_rate_pct', 'computed_poverty_rate', 'rate_difference']])

# Show top 10 counties
top_poverty_counties = child_poverty_validated.nlargest(10, 'child_poverty_rate_pct')
print("\nTop 10 counties by child poverty rate:")
print(top_poverty_counties[['NAME', 'state', 'children_in_poverty', 'total_children_under18', 'child_poverty_rate_pct']])

### Step 3: Ask AI to Create Visualization

**Your role:** Ask AI to create a clear, slide-ready visualization.

**Barebones prompt (copy-paste):**
```
Create a bar chart of top 10 counties by child poverty rate and save as toy-data/child_poverty_county.png.
```

**Expanded prompt:**
```
Create a horizontal bar chart showing the top 10 counties with highest child poverty rates. Make it slide-ready with clear labels, appropriate colors, and save as a PNG file suitable for presentations.
```

In [ ]:
# Create slide-ready visualization
plt.figure(figsize=(10, 6))

# Prepare data for plotting
plot_data = child_poverty.nlargest(10, 'child_poverty_rate_pct').copy()
plot_data['county_state'] = plot_data['NAME'] + ', ' + plot_data['state']
plot_data = plot_data.sort_values('child_poverty_rate_pct')

# Create horizontal bar chart
bars = plt.barh(plot_data['county_state'], plot_data['child_poverty_rate_pct'], 
                color='steelblue', alpha=0.8)

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, plot_data['child_poverty_rate_pct'])):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{value}%', ha='left', va='center', fontweight='bold')

# Formatting
plt.xlabel('Child Poverty Rate (%)', fontsize=12)
plt.title('Counties with Highest Child Poverty Rates', fontsize=14, fontweight='bold')
plt.suptitle('Top 10 counties across multiple states', fontsize=10, y=0.02)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

# Save plot
plt.savefig('toy-data/child_poverty_county.png', dpi=300, bbox_inches='tight')
plt.show()

print("Basic visualization saved successfully!")

### Step 4: Ask AI to Improve and Iterate

**Your role:** Ask AI to refine the analysis and visualization based on your needs.

**Barebones prompt (copy-paste):**
```
Improve legend and caption for slides.
```

**Expanded prompt:**
```
Improve the chart for a policy presentation. Add better formatting, consider color accessibility, and provide talking points about what the data shows for child welfare advocates.
```

In [ ]:
# Improved version with better accessibility and formatting
plt.figure(figsize=(12, 8))

# Prepare data with severity categories
plot_data = child_poverty.nlargest(10, 'child_poverty_rate_pct').copy()
plot_data['county_state'] = plot_data['NAME'] + ', ' + plot_data['state']
plot_data = plot_data.sort_values('child_poverty_rate_pct')

# Add severity categories
def categorize_severity(rate):
    if rate >= 35:
        return 'Very High (35%+)'
    elif rate >= 25:
        return 'High (25-34%)'
    elif rate >= 15:
        return 'Moderate (15-24%)'
    else:
        return 'Low (<15%)'

plot_data['severity'] = plot_data['child_poverty_rate_pct'].apply(categorize_severity)

# Color mapping for severity
color_map = {
    'Very High (35%+)': '#d73027',
    'High (25-34%)': '#fc8d59', 
    'Moderate (15-24%)': '#fee08b',
    'Low (<15%)': '#d9ef8b'
}

# Create the plot
colors = [color_map[severity] for severity in plot_data['severity']]
bars = plt.barh(plot_data['county_state'], plot_data['child_poverty_rate_pct'], 
                color=colors, alpha=0.9)

# Add value labels
for i, (bar, value) in enumerate(zip(bars, plot_data['child_poverty_rate_pct'])):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
             f'{value}%', ha='left', va='center', fontweight='bold', fontsize=11)

# Create custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color_map[cat], label=cat) for cat in color_map.keys()]
plt.legend(handles=legend_elements, title='Poverty Level', 
          loc='lower right', bbox_to_anchor=(1.0, 0.0))

# Formatting
plt.xlabel('Child Poverty Rate', fontsize=12)
plt.title('Child Poverty Crisis: Counties with Highest Rates', 
          fontsize=16, fontweight='bold', pad=20)
plt.figtext(0.5, 0.92, 'Over 40% of children live in poverty in some counties', 
            ha='center', fontsize=12, style='italic')
plt.figtext(0.5, 0.02, 
            'Data: Simulated county-level estimates for training purposes\n'
            'Note: Real analysis should use American Community Survey data', 
            ha='center', fontsize=9, style='italic')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

# Save improved version
plt.savefig('toy-data/child_poverty_county_improved.png', 
           dpi=300, bbox_inches='tight')
plt.show()

print("Improved visualization saved successfully!")

## Risks & Uncertainty

When working with AI on data analysis, always consider:

### Data Currency and Quality
- **Ask AI:** "List the columns you used and any assumptions you made about the data."
- **Verify:** Check data sources, collection dates, and methodology
- **Reality check:** These are toy numbers - real analysis requires current ACS data

### Statistical Limitations  
- **Ask AI:** "What are the limitations of this analysis and what caveats should I mention?"
- **Consider:** Margin of error, sample sizes, and confidence intervals
- **Small sample warning:** Some counties may have unreliable estimates due to small populations

### AI Analysis Limitations
- **Ask AI:** "What could be wrong with this analysis approach?"
- **Verify:** Cross-check AI suggestions against domain knowledge
- **Human judgment:** AI cannot interpret policy implications or recommend actions

### Recommended Verification Steps
1. **Column verification:** Always ask AI to report which specific columns it used
2. **Calculation check:** Verify that computed rates match provided rates  
3. **Outlier investigation:** Ask AI to flag and explain any unusual values
4. **Missing data:** Confirm how AI handled any missing or zero values

## Summary

This workflow demonstrates how humans and AI can collaborate effectively on data analysis:
- **Human provides context** and domain knowledge
- **AI handles technical implementation** and suggests improvements  
- **Human validates results** and makes final decisions
- **Both iterate together** to refine analysis and presentation

The key is maintaining human oversight while leveraging AI's speed and technical capabilities.